In [1]:
import duckdb
import pandas as pd

# Import jupysql Jupyter extension to create SQL cells
%load_ext sql

%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

In [2]:
%sql duckdb:///:memory:

In [4]:
%%sql
DROP TABLE IF EXISTS sales;
DROP TABLE IF EXISTS customers;
DROP TABLE IF EXISTS marketing;

CREATE TABLE sales AS SELECT * FROM read_csv_auto('tables/sales.csv');
CREATE TABLE customers AS SELECT * FROM read_csv_auto('tables/customers.csv');
CREATE TABLE marketing AS SELECT * FROM read_csv_auto('tables/marketing.csv');

,Success


In [5]:
%sql DESCRIBE

,Success


## Example
Tell me all the product names.

In [6]:
%%sql

select
    item
from
    sales
group by
    item

,item
0,Bed
1,Sofa
2,Dinner table


In [42]:
%%sql
    select * from  marketing;

,date,customer_email,channel
0,2025-10-15,iyoung@farrell-mills.com,Referral
1,2025-12-13,tcervantes@pope.biz,Unknown
2,2025-12-02,alexamccullough@barker.com,Unknown
3,2025-12-02,dcarter@yahoo.com,Direct
4,2025-09-24,eugeneramirez@perez-mendez.com,Organic
...,...,...,...
99995,2025-09-23,jonathanfloyd@burnett.com,Referral
99996,2026-01-30,jessicagarcia@richardson-savage.net,Email
99997,2025-11-03,wrich@gmail.com,Organic
99998,2025-12-18,woodscarrie@gmail.com,Unknown


## Question 1
Tell me the top 10 countries by their number of sales.

In [37]:
%%sql

select c.country, count(*) as Total_sales
    from sales s 
    inner join customers c on s.customer_id = c.id
group by country
order by Total_sales desc
Limit 10;

,country,Total_sales
0,Germany,2202
1,USA,2089
2,Denmark,1095
3,France,771
4,Canada,704
5,Austria,681
6,UK,628
7,Sweden,440
8,Spain,363
9,Norway,345


## Question 1.5
Tell me the top 9 countries by sales, plus a 10th row for “all others”.

In [40]:
%%sql
WITH country_sales AS (
    SELECT 
        c.country,
        COUNT(*) AS total_sales
    FROM sales s
    JOIN customers c 
        ON s.customer_id = c.id
    GROUP BY c.country
),
ranked AS (
    SELECT 
        country,
        total_sales,
        ROW_NUMBER() OVER (ORDER BY total_sales DESC) AS rn
    FROM country_sales
)

SELECT 
    CASE 
        WHEN rn <= 9 THEN country
        ELSE 'all others'
    END AS country,
    SUM(total_sales) AS total_sales
FROM ranked
GROUP BY 
    CASE 
        WHEN rn <= 9 THEN country
        ELSE 'all others'
    END
ORDER BY total_sales DESC;

,country,total_sales
0,Germany,2202.0
1,USA,2089.0
2,Denmark,1095.0
3,all others,1027.0
4,France,771.0
5,Canada,704.0
6,Austria,681.0
7,UK,628.0
8,Sweden,440.0
9,Spain,363.0


## Question 2
A repeat customer is someone who makes 2+ purchases, but 2+ on the same day doesn’t count. Tell me how many repeat customers we have.

In [ ]:
%%sql

SELECT COUNT(*) AS repeat_customers
FROM (
    SELECT customer_id
    FROM sales
    GROUP BY customer_id
    HAVING COUNT(DISTINCT date) >= 2
) t;

## Question 3
Tell me the count of customers with only organic orders on a last touch attribution basis.

In [46]:
%%sql
WITH last_touch AS (
    SELECT
        s.customer_id,
        s.date AS sale_date,
        m.channel,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id, s.date
            ORDER BY m.date DESC
        ) AS rn
    FROM sales s
    JOIN customers c 
        ON s.customer_id = c.id
    LEFT JOIN marketing m
        ON c.email = m.customer_email
        AND m.date <= s.date
)

SELECT COUNT(*) AS customers_with_only_organic
FROM (
    SELECT customer_id
    FROM last_touch
    WHERE rn = 1
    GROUP BY customer_id
    HAVING COUNT(*) = COUNT(CASE WHEN channel = 'Organic' THEN 1 END)
) t;

,customers_with_only_organic
0,886
